# Step 4: Generate Submission — Demo Notebook
---
**Goal**: Apply feature engineering to test set, predict with trained XGBoost, and export submission.csv.

### Process:
1. Load test data + trained model
2. Replicate feature engineering (concatenate train+test for lags)
3. Predict sales for 2017-08-16 to 2017-08-31
4. Validate submission format
5. Export submission.csv

In [ ]:
import sys, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xgboost as xgb

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

sys.path.insert(0, os.path.abspath(''))
from config import *
from utils import Timer, rmsle

print('Libraries loaded.')

In [ ]:
# Load trained model and feature list
model_path = os.path.join(MODELS_DIR, 'xgboost_model.json')
booster = xgb.Booster()
booster.load_model(model_path)

# Load feature list
feature_list_path = os.path.join(MODELS_DIR, 'feature_list.txt')
with open(feature_list_path, 'r') as f:
    feature_cols = [line.strip() for line in f if line.strip()]

print(f'Model loaded: {model_path}')
print(f'Features: {len(feature_cols)}')

In [ ]:
# Load the generated submission
submission_path = os.path.join(SUBMISSIONS_DIR, 'submission.csv')
submission = pd.read_csv(submission_path)

print(f'Submission: {submission.shape[0]:,} rows')
print(f'Columns: {submission.columns.tolist()}')
print(f'\nSales statistics:')
print(f'  Mean:   {submission["sales"].mean():.2f}')
print(f'  Median: {submission["sales"].median():.2f}')
print(f'  Min:    {submission["sales"].min():.2f}')
print(f'  Max:    {submission["sales"].max():.2f}')
print(f'  Sum:    {submission["sales"].sum():,.0f}')
print(f'  Zeros:  {(submission["sales"] == 0).sum():,}')
submission.head(10)

## 1. Prediction Distribution vs Training Distribution

In [ ]:
# Compare prediction distribution with training distribution
df_train = pd.read_csv(TRAIN_CLEANED_PATH, parse_dates=['date'])
train_sales = df_train[df_train['date'] >= '2017-01-01']['sales']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
axes[0].hist(train_sales, bins=100, alpha=0.5, label='Train (2017)', color='#3498DB', density=True)
axes[0].hist(submission['sales'], bins=100, alpha=0.5, label='Predictions', color='#E74C3C', density=True)
axes[0].set_xlim(0, 2000)
axes[0].set_title('Sales Distribution: Training vs Predictions', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Sales')
axes[0].set_ylabel('Density')
axes[0].legend()

# CDF comparison
axes[1].hist(train_sales, bins=500, alpha=0.5, label='Train (2017)', color='#3498DB',
             cumulative=True, density=True, histtype='step', linewidth=2)
axes[1].hist(submission['sales'], bins=500, alpha=0.5, label='Predictions', color='#E74C3C',
             cumulative=True, density=True, histtype='step', linewidth=2)
axes[1].set_xlim(0, 2000)
axes[1].set_title('Cumulative Distribution: Training vs Predictions', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sales')
axes[1].set_ylabel('Cumulative Probability')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Submission Format Validation

Verify the submission matches Kaggle's expected format (id, sales).

In [ ]:
# Load sample submission for comparison
df_sample = pd.read_csv(SAMPLE_SUB_PATH)

print('Format validation:')
print(f'  Same row count: {len(submission) == len(df_sample)} (expected {len(df_sample)})')
print(f'  Same columns:   {list(submission.columns) == list(df_sample.columns)}')
print(f'  ID range match: {submission["id"].min() == df_sample["id"].min()} ~ {submission["id"].max() == df_sample["id"].max()}')

# Compare with naive baseline (predict 0 for everything)
naive_rmsle = rmsle(df_sample['sales'].values * 0 + 1, df_sample['sales'].values + 1)
print(f'\n  Naive RMSLE (predict mean): ~1.0+')
print(f'  Our model RMSLE (on validation): ~0.236')
print(f'\n  Sample submission (all zeros): mean={df_sample["sales"].mean()}')
print(f'  Our submission:                 mean={submission["sales"].mean():.2f}')

---
## Project Complete!

### 4-Step Pipeline Summary

| Step | Script | Notebook | Output |
|------|--------|----------|--------|
| 1. EDA + Cleaning | `step1_eda_cleaning.py` | `step1_visualization.ipynb` | `data/processed/` |
| 2. Feature Engineering | `step2_feature_engineering.py` | `step2_feature_engineering.ipynb` | `data/features/` |
| 3. XGBoost Training | `step3_model_training.py` | `step3_model_training.ipynb` | `models/` |
| 4. Submission | `step4_generate_submission.py` | `step4_generate_submission.ipynb` | `submissions/submission.csv` |

### Key Results
- **163 features** engineered: trend (LR), seasonal (Fourier), lags, rolling, holidays, oil
- **XGBoost Validation RMSLE: ~0.236**
- **Top features**: rolling_mean_7d, rolling_mean_14d, sales_lag_7
- **Submission file**: `submissions/submission.csv` — ready for Kaggle upload

### Next Steps for Improvement
- Add more granular holiday matching (Regional/Local by city/state)
- Include transactions as a feature with its own forecast model
- Try CatBoost or LightGBM for comparison
- Ensemble multiple models
- Add external data (weather, economic indicators)